# Task 06: Kiểm định nghiệm đơn vị panel (Unit Root Tests)

**Mục tiêu:** Kiểm tra tính dừng của dữ liệu panel trước khi chạy hồi quy

**Các biến cần kiểm tra:**
- GDP_growth (Biến phụ thuộc)
- E_score (Environmental score)
- S_score (Social score)
- G_score (Governance score)
- inflation (Lạm phát)
- population_growth (Tăng trưởng dân số)

**Các test sẽ sử dụng:**
1. Levin-Lin-Chu (LLC) - Common unit root process
2. Im-Pesaran-Shin (IPS) - Individual unit root process
3. ADF-Fisher - Individual unit root process

In [ ]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS, PooledOLS
from linearmodels.panel import PanelUnitRoot
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/processed/asean-esg-final.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nSố lượng quốc gia: {df['ISO3 code'].nunique()}")
print(f"Khoảng thời gian: {df['year'].min()} - {df['year'].max()}")
df.head()

In [ ]:
# Chuẩn bị dữ liệu cho panel analysis
# Set multi-index: entity (country) and time (year)
df_panel = df.set_index(['ISO3 code', 'year'])
df_panel = df_panel.sort_index()

# Các biến cần test
variables_to_test = ['GDP_growth', 'E_score', 'S_score', 'G_score', 
                     'inflation', 'population_growth']

# Kiểm tra missing values
print("Missing values per variable:")
print(df_panel[variables_to_test].isnull().sum())

## 1. Levin-Lin-Chu (LLC) Test

H0: Tất cả các panel có unit root (non-stationary)
H1: Tất cả các panel stationary

In [ ]:
from linearmodels.panel import PanelUnitRoot

# LLC test cho từng biến
llc_results = {}

for var in variables_to_test:
    try:
        # Loại bỏ missing values
        data = df_panel[var].dropna()
        
        # LLC test
        test = PanelUnitRoot(data, test_type='llc')
        stat = test.stat
        pval = test.pvalue
        
        llc_results[var] = {
            'statistic': stat,
            'pvalue': pval,
            'stationary': 'Yes' if pval < 0.05 else 'No'
        }
        
        print(f"\n{var}:")
        print(f"  LLC statistic: {stat:.4f}")
        print(f"  p-value: {pval:.4f}")
        print(f"  Stationary (5%): {'Yes' if pval < 0.05 else 'No'}")
        
    except Exception as e:
        print(f"\n{var}: Error - {str(e)}")
        llc_results[var] = {'statistic': np.nan, 'pvalue': np.nan, 'stationary': 'Error'}

## 2. Im-Pesaran-Shin (IPS) Test

H0: Tất cả các panel có unit root
H1: Ít nhất một panel stationary (individual effects)

In [ ]:
# IPS test cho từng biến
ips_results = {}

for var in variables_to_test:
    try:
        data = df_panel[var].dropna()
        
        # IPS test
        test = PanelUnitRoot(data, test_type='ips')
        stat = test.stat
        pval = test.pvalue
        
        ips_results[var] = {
            'statistic': stat,
            'pvalue': pval,
            'stationary': 'Yes' if pval < 0.05 else 'No'
        }
        
        print(f"\n{var}:")
        print(f"  IPS statistic: {stat:.4f}")
        print(f"  p-value: {pval:.4f}")
        print(f"  Stationary (5%): {'Yes' if pval < 0.05 else 'No'}")
        
    except Exception as e:
        print(f"\n{var}: Error - {str(e)}")
        ips_results[var] = {'statistic': np.nan, 'pvalue': np.nan, 'stationary': 'Error'}

## 3. ADF-Fisher Test

H0: Tất cả các panel có unit root
H1: Ít nhất một panel stationary

In [ ]:
# ADF-Fisher test cho từng biến
adf_results = {}

for var in variables_to_test:
    try:
        data = df_panel[var].dropna()
        
        # ADF-Fisher test
        test = PanelUnitRoot(data, test_type='adf')
        stat = test.stat
        pval = test.pvalue
        
        adf_results[var] = {
            'statistic': stat,
            'pvalue': pval,
            'stationary': 'Yes' if pval < 0.05 else 'No'
        }
        
        print(f"\n{var}:")
        print(f"  ADF-Fisher statistic: {stat:.4f}")
        print(f"  p-value: {pval:.4f}")
        print(f"  Stationary (5%): {'Yes' if pval < 0.05 else 'No'}")
        
    except Exception as e:
        print(f"\n{var}: Error - {str(e)}")
        adf_results[var] = {'statistic': np.nan, 'pvalue': np.nan, 'stationary': 'Error'}

## 4. Tổng hợp kết quả

In [ ]:
# Tạo bảng tổng hợp
results_table = []

for var in variables_to_test:
    llc = llc_results.get(var, {})
    ips = ips_results.get(var, {})
    adf = adf_results.get(var, {})
    
    # Đếm số test cho kết luận stationary
    stationary_count = sum([
        llc.get('stationary') == 'Yes',
        ips.get('stationary') == 'Yes',
        adf.get('stationary') == 'Yes'
    ])
    
    # Kết luận: majority vote
    conclusion = 'I(0)' if stationary_count >= 2 else 'I(1)'
    
    results_table.append({
        'Variable': var,
        'LLC_Stat': llc.get('statistic', np.nan),
        'LLC_pval': llc.get('pvalue', np.nan),
        'IPS_Stat': ips.get('statistic', np.nan),
        'IPS_pval': ips.get('pvalue', np.nan),
        'ADF_Stat': adf.get('statistic', np.nan),
        'ADF_pval': adf.get('pvalue', np.nan),
        'Conclusion': conclusion
    })

df_results = pd.DataFrame(results_table)
print("\n" + "="*80)
print("BẢNG KẾT QUẢ KIỂM ĐỊNH NGHIỆM ĐƠN VỊ PANEL")
print("="*80)
print(df_results.to_string(index=False))
print("\nGhi chú:")
print("- I(0): Stationary at level (dùng giá trị gốc)")
print("- I(1): Non-stationary at level (cần sai phân bậc 1)")

## 5. Tạo biến sai phân (nếu cần)

In [ ]:
# Tạo biến sai phân cho các biến I(1)
i1_vars = df_results[df_results['Conclusion'] == 'I(1)']['Variable'].tolist()

if len(i1_vars) > 0:
    print(f"\nCác biến cần sai phân bậc 1: {i1_vars}")
    
    for var in i1_vars:
        # Tạo biến sai phân
        df_panel[f'd_{var}'] = df_panel.groupby(level=0)[var].diff()
        print(f"  Tạo biến d_{var}")
    
    # Lưu dataset mới
    df_panel_reset = df_panel.reset_index()
    df_panel_reset.to_csv('../data/processed/asean-esg-with-diff.csv', index=False)
    print(f"\nĐã lưu dataset với biến sai phân: data/processed/asean-esg-with-diff.csv")
else:
    print("\nTất cả biến đều stationary (I(0)). Không cần tạo biến sai phân.")

## 6. Lưu kết quả ra LaTeX table

In [ ]:
# Tạo LaTeX table
latex_content = """\\begin{table}[htbp]
\\centering
\\caption{Kết quả kiểm định nghiệm đơn vị panel}
\\label{tab:unit_root}
\\small
\\begin{tabular}{lrrrrrrl}
\\hline
\\hline
Biến & LLC & p-value & IPS & p-value & ADF & p-value & Kết luận \\\\
\\hline
"""

for _, row in df_results.iterrows():
    latex_content += f"{row['Variable']} & {row['LLC_Stat']:.3f} & {row['LLC_pval']:.3f} & "
    latex_content += f"{row['IPS_Stat']:.3f} & {row['IPS_pval']:.3f} & "
    latex_content += f"{row['ADF_Stat']:.3f} & {row['ADF_pval']:.3f} & {row['Conclusion']} \\\\\n"

latex_content += """\\hline
\\hline
\\end{tabular}
\\begin{tablenotes}
\\small
\\item Ghi chú: LLC = Levin-Lin-Chu, IPS = Im-Pesaran-Shin, ADF = ADF-Fisher. \\
\\item I(0) = Stationary at level, I(1) = Non-stationary (cần sai phân). \\
\\item Mức ý nghĩa: 5\\%. N = 563 quan sát.
\\end{tablenotes}
\\end{table}
"""

# Lưu file
with open('../LaTeX/tables/table-03-unit-root.tex', 'w', encoding='utf-8') as f:
    f.write(latex_content)

print("\nĐã lưu bảng kết quả: LaTeX/tables/table-03-unit-root.tex")
print("\n" + latex_content)

In [ ]:
# Summary statistics
print("\n" + "="*80)
print("TỔNG KẾT")
print("="*80)
print(f"\nSố biến I(0) (stationary): {len(df_results[df_results['Conclusion'] == 'I(0)'])}")
print(f"Số biến I(1) (non-stationary): {len(df_results[df_results['Conclusion'] == 'I(1)'])}")
print("\nBiến stationary (I(0)):")
print(df_results[df_results['Conclusion'] == 'I(0)']['Variable'].tolist())
print("\nBiến non-stationary (I(1)):")
print(df_results[df_results['Conclusion'] == 'I(1)']['Variable'].tolist())